In [1]:
import pandas as pd
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# # Columns to be encoded by frequency
freq_cols = [
    'album_name',
    'artists',
    'track_genre'
]

# columns to be deleted
drop_cols = [
    'track_id',
    'track_name',   
    'loudness',     
    'acousticness',
    'mode',
    'speechiness',
    'time_signature',
    'key',
    'explicit'
]


class ColumnDropper(BaseEstimator, TransformerMixin):
    def __init__(self, drop_cols):
        self.drop_cols = drop_cols
    def fit(self, X, y = None):
        return self
    def transform(self, X, y = None):
        self.X = X.copy()
        self.common_cols = list(filter(lambda com_col : com_col in self.drop_cols, self.X.columns))
        return self.X.drop(columns = self.common_cols)

class DurationConverter(BaseEstimator, TransformerMixin):
    def fit(self, X, y = None):
        return self
    def transform(self, X, y=None):
        self.X = X.copy()
        if 'duration_ms' not in self.X:
            return self.X
        def convert_ms_to_cat(ms):
            if ms <= 120000: return 1
            elif ms <= 360000: return 2
            elif ms <= 1200000: return 3
            elif ms <= 2400000: return 4
            else: return 5
        self.X['duration_ms'] = self.X['duration_ms'].apply(convert_ms_to_cat)
        return self.X    

class FrequencyEncoder(BaseEstimator, TransformerMixin): 
    def __init__(self, freq_cols):
        self.freq_cols = freq_cols
        self.freq_map = {}
    def fit(self, X, y = None):
        for col in self.freq_cols:
            self.freq_map[col] = X[col].value_counts()
        return self
    def transform(self, X, y = None):
        self.X = X.copy()
        self.active_cols = list(filter(lambda com_col : com_col in self.freq_cols, self.X.columns))
        for col in self.active_cols:
            self.X[f"{col}_pop"] = self.X[col].map(self.freq_map[col])
            self.X[f"{col}_pop"] = self.X[f"{col}_pop"].fillna(0.00001)
        self.X = self.X.drop(columns = self.active_cols)
        return self.X

In [3]:
# load all pipelines
pipeline = joblib.load('pipeline.joblib')
vectors = joblib.load('vectors.joblib')

In [6]:
df = pd.read_csv("cleaned_data.csv")

In [7]:
def recommended_songs(input_song, n = 6):
    input_df = pd.DataFrame([input_song])
    assigned_vector = pipeline[:-1].transform(input_df)
    assigned_cluster = pipeline.predict(input_df)[0]
    
    cluster_vectors = vectors[vectors['cluster'] == assigned_cluster].iloc[:, :-1]
    cluster_masks = cluster_vectors.select_dtypes(include='number')
    similarities = cosine_similarity(
        cluster_masks, assigned_vector
    )
    sim_scores = sorted(zip(cluster_masks.index, similarities),
        key = lambda x : x[1],
        reverse = True
    )[:n]
    indices = [ss[0] for ss in sim_scores]
    scores  = [round(s[1].item(), 5) for s in sim_scores]
    pred_songs = cluster_vectors.loc[indices, ['track_id', 'track_name', 'artists', 'track_genre', 'album_name']].copy()
    pred_songs['similarity'] = scores
    return pred_songs

In [8]:
def findRecomendation(song_name, n = 6):
	matched_song = df[df['track_name'] == song_name]
	recomendation = pd.DataFrame()
	if matched_song.empty:
		return recomendation
	matched_song_dict = matched_song.to_dict(orient = 'records')
	matched_song = matched_song[['track_id', 'track_name', 'artists', 'track_genre', 'album_name']]
	for input_song in matched_song_dict:
		recomendation = pd.concat([recomendation, recommended_songs(input_song, n)])
	recomendation = recomendation[~recomendation['track_id'].isin(matched_song['track_id'])]
	recomendation = recomendation.sort_values(by = 'similarity', ascending = False)[:n]
	recomendation = recomendation.reset_index(drop = True)
	recomendation.index += 1
	return recomendation, matched_song.iloc[0]

In [10]:
recomended_songs, given_song = findRecomendation("Love Paradise")

In [13]:
recomended_songs

,track_id,track_name,artists,track_genre,album_name,similarity
1,0bqYvGR4vP2KstdMyemKYI,doodoodoo,Terence Lam,cantopop,doodoodoo,0.99868
2,11IqNbLOD4s4nVYSuEttFR,"Dear My Friend,",Keung To,cantopop,"Dear My Friend,",0.99666
3,5ntMWSjQGsHb4TIksSIUBc,The Best Is Yet To Come,At17,cantopop,Threesome,0.99569
4,7CXIxU4GflzqTiaRNfS6LY,囚鳥 - Remastered,Cass Phang,cantopop,囚鳥 (Remastered)),0.99528
5,2KogAeAixsmxIzDmMHKhjY,下一位前度 in Ab major,Terence Lam,cantopop,MAJOR IN MINOR,0.99313


In [14]:
given_song

track_id       73LRTBKmdjroFDSBGSqClt
track_name              Love Paradise
artists                    Kelly Chen
track_genre                  cantopop
album_name        Kelly Stylish Index
Name: 11610, dtype: object

In [19]:
# song passed to 
input_song = {
    'track_id':         '32QnXosZq7A11knnBAEqk7',
    'artists':          'Arijit Singh',
    'album_name':       'Aashiqui 2',
    'track_name':       'Tum Hi Ho',
    'popularity':        80,
    'duration_ms':       234000,
    'explicit':          0,
    'danceability':      0.4,
    'energy':            0.3,
    'key':               4,
    'loudness':         -8.5,
    'mode':              1,
    'speechiness':       0.05,
    'acousticness':      0.8,
    'instrumentalness':  0.0,
    'liveness':          0.1,
    'valence':           0.3,
    'tempo':             72.0,
    'time_signature':    4,
    'track_genre':      'bollywood'
}

,track_id,track_name,artists,track_genre,album_name,similarity
1,73LRTBKmdjroFDSBGSqClt,Love Paradise,Kelly Chen,cantopop,Kelly Stylish Index,0.99998
2,0bqYvGR4vP2KstdMyemKYI,doodoodoo,Terence Lam,cantopop,doodoodoo,0.99998
3,2KogAeAixsmxIzDmMHKhjY,下一位前度 in Ab major,Terence Lam,cantopop,MAJOR IN MINOR,0.99998
4,5PZJxhLQ34IbwrB6VDVSnz,Wonderful tonight,Khalil Fong,cantopop,Timeless,0.99998
5,11IqNbLOD4s4nVYSuEttFR,"Dear My Friend,",Keung To,cantopop,"Dear My Friend,",0.99998
